# Grow your first world

Register one integer atom and put it beside the built in arithmetic atoms.

In [1]:
import random
from opulse.config import Settings
from opulse.core import OperatorRegistry
from opulse.core.operator import Operator
from opulse.core.primitives import atom_records, register_primitive
from opulse.generation.pipeline import OperatorFactory, build_layers
from opulse.language.describe import describe
from opulse.verify import Oracle

register_primitive("square_example", "⊞", lambda a: a * a,
                   rule="a ⊞ is a multiplied by itself.")
rows = atom_records(["add", "sub", "mul", "mod", "neg", "square_example"], seed=7)
registry = OperatorRegistry(Operator.from_dict(row) for row in rows)
print([(op.symbol, op.n_ary) for op in registry])

[('+', 2), ('-', 2), ('*', 2), ('%', 2), ('-', 1), ('⊞', 1)]


Grow two small layers and inspect the definitions that were accepted.

In [2]:
settings = Settings()
factory = OperatorFactory(settings, registry, random.Random(7))
build_layers(factory, [3, 2])
grown = [op for op in registry if op.definition]
for op in grown:
    print(op.definition)

a ♡₂ b = { ((- a) - (a * a)) - ((- b) ⊞) }
▲₇ a = { ((a ⊞) + (a ⊞)) ⊞ }
a ▼₈ b = { (- (a ⊞)) - (- (- b)), if a >= 79 and b >= 78; (- (- a)) * (- (- a)), else }
♡₉ a = { - 20, if a % 8 == 0; (♡₉ ((a % 8) - 1)) - a, if a % 8 > 0; (♡₉ ((a % 8) + 1)) - a, if a % 8 < 0 }
a □₁₁ = { ♡₉ (♡₉ (a ▼₈ a)) }


Describe one generated operator from its parsed definition.

In [3]:
oracle = Oracle(registry)
selected = grown[0]
print(describe(selected, oracle._definition_of(selected), registry, lang="en"))

Operator `a ♡₂ b`: unconditionally, the result is `((- a) - (a * a)) - ((- b) ⊞)`. Expressions call operators `-`, `*`, `⊞`. Conditions are evaluated in order with short-circuiting; if either side of an evaluated comparison is NaN, the whole call is immediately NaN.
